<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/Allison/MLTest_CorrelationData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

drive.mount('/content/drive')

#Loading Data
path = "/content/drive/MyDrive/sparcs_cleaned_v3.csv"
data = pd.read_csv(path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Looking at the Dataset Info
print(data.shape)
data.info()
data.head()

(4238636, 30)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4238636 entries, 0 to 4238635
Data columns (total 30 columns):
 #   Column                            Dtype  
---  ------                            -----  
 0   Health Service Area               object 
 1   Hospital County                   object 
 2   Permanent Facility Id             float64
 3   Age Group                         object 
 4   Zip Code                          object 
 5   Gender                            object 
 6   Race                              object 
 7   Ethnicity                         object 
 8   Length of Stay                    object 
 9   Type of Admission                 object 
 10  Patient Disposition               object 
 11  Discharge Year                    int64  
 12  CCSR Diagnosis Code               object 
 13  CCSR Procedure Code               object 
 14  APR DRG Code                      int64  
 15  APR MDC Code                      int64  
 16  APR Severity of Illnes

,Health Service Area,Hospital County,Permanent Facility Id,Age Group,Zip Code,Gender,Race,Ethnicity,Length of Stay,Type of Admission,...,Medicaid,Medicare,Private Health Insurance,Self-Pay,Blue Cross/Blue Shield,Miscellaneous/Other,Federal/State/Local/VA,Department of Corrections,"Managed Care, Unspecified",Number of Payment Typologies
0,New York City,Bronx,3058.0,50-69,104,F,Other Race,Spanish/Hispanic,1,Emergency,...,1,0,0,0,0,0,0,0,0,1
1,New York City,Bronx,1168.0,30-49,104,M,Black/African American,Not Span/Hispanic,4,Emergency,...,1,0,0,0,0,0,0,0,0,1
2,New York City,Bronx,3058.0,50-69,104,M,Other Race,Not Span/Hispanic,4,Emergency,...,1,1,0,0,0,0,0,0,0,2
3,New York City,Bronx,1169.0,18-29,104,M,Black/African American,Not Span/Hispanic,5,Emergency,...,1,0,0,0,0,0,0,0,0,1
4,New York City,Bronx,1169.0,50-69,104,F,Other Race,Spanish/Hispanic,3,Emergency,...,1,1,0,0,0,0,0,0,0,2


In [ ]:
#Looking at columns, which data would be useful to be included
print(data.columns.tolist())

['Health Service Area', 'Hospital County', 'Permanent Facility Id', 'Age Group', 'Zip Code', 'Gender', 'Race', 'Ethnicity', 'Length of Stay', 'Type of Admission', 'Patient Disposition', 'Discharge Year', 'CCSR Diagnosis Code', 'CCSR Procedure Code', 'APR DRG Code', 'APR MDC Code', 'APR Severity of Illness Code', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Total Charges', 'Medicaid', 'Medicare', 'Private Health Insurance', 'Self-Pay', 'Blue Cross/Blue Shield', 'Miscellaneous/Other', 'Federal/State/Local/VA', 'Department of Corrections', 'Managed Care, Unspecified', 'Number of Payment Typologies']


In [ ]:
#Looking at ZIP Codes
print("Unique ZIP codes:", data['Zip Code'].nunique())
print(sorted(data['Zip Code'].unique()))


Unique ZIP codes: 50
['100', '101', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', 'OOS']


In [ ]:
drop_columns=['Health Service Area', 'Hospital County', 'Permanent Facility Id',
              'Patient Disposition', 'Discharge Year', 'CCSR Diagnosis Code',
              'CCSR Procedure Code', 'APR DRG Code', 'APR MDC Code']

In [ ]:
#Creating a function to clean and prepare data set
def clean_data(df):

#Remove Duplicates
  df = df.drop_duplicates()

#Excluding the rows that do not have LOS; Missing data
  df = df.dropna(subset = ['Length of Stay'])

#Convert data to numerical, Drop those who can't convert
  df['Length of Stay'] = pd.to_numeric(df['Length of Stay'], errors='coerce')
  df = df.dropna(subset = ['Length of Stay'])

#Remove out-of-state ZIP Codes ('OOS')
  if 'Zip Code' in df.columns:
      df = df[df['Zip Code'] != 'OOS']

  df = df.drop(columns =[c for c in drop_columns if c in df.columns], errors = 'ignore')

  return df

#Clean Data
data = clean_data(data)

print("Shape of cleaned data:", data.shape)  # rows, columns




Shape of cleaned data: (4074873, 21)


In [ ]:
numerical_columns = ['Medicare','Private Health Insurance','Self-Pay', 'Blue Cross/Blue Shield',
                     'Miscellaneous/Other', 'Federal/State/Local/VA', 'Department of Corrections',
                     'Managed Care, Unspecified', 'Number of Payment Typologies']

categorical_columns = ['Zip Code', 'Age Group', 'Gender', 'Race', 'Ethnicity',
                       'Type of Admission', 'APR Risk of Mortality']




In [ ]:
#CORRELATIONS FOR NUMERICAL DATA (INT64)

#Getting numerical columns (int + float)
numerical_columns = data.select_dtypes(include = ['int64', 'float64']).columns.tolist()

#Remove target variable from the predictor list
numerical_columns.remove('Length of Stay')
display(data[numerical_columns].describe())

#Calculate correlations between LOS and each numeric variable
correlations = (data[numerical_columns + ['Length of Stay']]
                .corr()['Length of Stay'].sort_values(ascending = False))
print(correlations)



#DATA TAKEAWAYS

#Medicare - Slight positive correlation: patients with Medicare tend to have slightly longer stays. But still not very strong.
#More payment types associated with slightly longer stay, could indicate complex cases

#Payment indicators have very weak correlation with LOS, are not strong predictors


,APR Severity of Illness Code,Total Charges,Medicaid,Medicare,Private Health Insurance,Self-Pay,Blue Cross/Blue Shield,Miscellaneous/Other,Federal/State/Local/VA,Department of Corrections,"Managed Care, Unspecified",Number of Payment Typologies
count,4.074873e+06,4.074873e+06,4.074873e+06,4.074873e+06,4.074873e+06,4.074873e+06,4.074873e+06,4.074873e+06,4.074873e+06,4.074873e+06,4.074873e+06,4.074873e+06
mean,2.150070e+00,8.390861e+04,4.655961e-01,4.400795e-01,2.100156e-01,1.343880e-01,1.470407e-01,1.301317e-02,1.646260e-02,7.443177e-04,1.665205e-02,1.591821e+00
std,9.357241e-01,1.478129e+05,4.988150e-01,4.963966e-01,4.073194e-01,3.410687e-01,3.541465e-01,1.133306e-01,1.272462e-01,2.727203e-02,1.279639e-01,6.980662e-01
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,2.289959e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
50%,2.000000e+00,4.540815e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
75%,3.000000e+00,9.030752e+04,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00
max,4.000000e+00,1.798104e+07,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,3.000000e+00


Length of Stay                  1.000000
Total Charges                   0.658771
APR Severity of Illness Code    0.350156
Medicare                        0.102859
Number of Payment Typologies    0.065181
Medicaid                        0.056008
Miscellaneous/Other             0.006901
Department of Corrections       0.001560
Federal/State/Local/VA         -0.001369
Managed Care, Unspecified      -0.011167
Self-Pay                       -0.011857
Private Health Insurance       -0.044266
Blue Cross/Blue Shield         -0.045726
Name: Length of Stay, dtype: float64


In [ ]:
#CORRELATIONS FOR CATERGORICAL DATA (object)

categorical_columns = ['Zip Code', 'Age Group', 'Gender', 'Race', 'Ethnicity',
                       'Type of Admission', 'APR Risk of Mortality']

#One-hot encoded
data_encoded = pd.get_dummies(data[categorical_columns], drop_first=True)

#Add LOS for correlation
data_encoded['Length of Stay'] = data['Length of Stay']

#Calculate correlations between LOS and all one-hot encoded categorical variables
correlations_cat = data_encoded.corr()['Length of Stay'].sort_values(ascending=False)

print(correlations_cat)

#mean LOS per category, which categories have longer or shorter stays
for col in categorical_columns:
    print(f"\nAverage LOS by {col}:")
    print(data.groupby(col)['Length of Stay'].mean().sort_values(ascending=False).head(10))




#DATA TAKEAWAYS

#APR Risk of Mortality show the strongest relationship with LOS, especially:
#Extreme ~12.6 days on average, Minor ~3.9 days on average
#LOS increases sharply with mortality risk, which is expected clinically.

#Age Group shows older patients have longer stays:
#70 or Older 6.48 days
#0-17 3.92 days

#Race, some differences in LOS:
#Black/African American	6.62 days
#Multi-racial 5.61 days
#White	5.56 days
#Other Race	5.37 days

#ZIP code: some variation by area, but weaker effects



Length of Stay                    1.000000
APR Risk of Mortality_Major       0.121948
Type of Admission_Emergency       0.078026
Age Group_70 or Older             0.068375
Gender_M                          0.061639
                                    ...   
APR Risk of Mortality_Moderate   -0.030300
Age Group_18-29                  -0.041044
Age Group_30-49                  -0.042428
Type of Admission_Newborn        -0.083860
APR Risk of Mortality_Minor      -0.222270
Name: Length of Stay, Length: 70, dtype: float64

Average LOS by Zip Code:
Zip Code
146    6.761973
139    6.447123
116    6.354984
126    6.327925
143    6.276860
132    6.267271
107    6.173549
108    6.130765
103    6.124475
142    6.086775
Name: Length of Stay, dtype: float64

Average LOS by Age Group:
Age Group
70 or Older    6.475707
50-69          6.464313
30-49          5.004362
18-29          4.616648
0-17           3.893357
Name: Length of Stay, dtype: float64

Average LOS by Gender:
Gender
M    6.229568
F    5.